# Lab 3: Histogram - Atomic Operations

---

## Overview

This lab explores the **histogram problem**, a classic example that demonstrates the challenges of parallel programming when multiple threads need to update shared data. You will learn how **atomic operations** solve race conditions and how **shared memory privatization** dramatically improves performance.

Building on Lab 2's introduction to shared memory for data movement (matrix transpose), this lab shows how shared memory reduces **atomic contention** in accumulation patterns.

Histograms are fundamental to many algorithms including image processing, data analysis, and machine learning preprocessing.

## Learning Objectives

By the end of this lab, you will be able to:

1. Identify race conditions in parallel programs
2. Use atomic operations (`atomicAdd`) to safely update shared data
3. Understand the performance cost of atomic contention
4. Implement two-level histogram reduction using shared memory
5. Analyze memory access patterns and their impact on performance

## 1. The Histogram Problem

### Sequential Algorithm

The serial implementation is straightforward:

```cpp
void serial_histogram(const int* input, int* histogram, int N, int num_bins) {
    // Initialize histogram to zero
    for (int i = 0; i < num_bins; i++) {
        histogram[i] = 0;
    }
    // Count occurrences
    for (int i = 0; i < N; i++) {
        histogram[input[i]]++;
    }
}
```

### The Parallelization Challenge

When multiple threads try to increment the same histogram bin simultaneously, we have a **race condition**:

```
Thread A reads histogram[5] = 10
Thread B reads histogram[5] = 10
Thread A writes histogram[5] = 11
Thread B writes histogram[5] = 11  <-- Lost update!
```

The correct value should be 12, but we get 11. This is called a **lost update**.

## 2. Atomic Operations

**Atomic operations** guarantee that a read-modify-write sequence completes without interruption from other threads.

### atomicAdd

```cpp
atomicAdd(&histogram[bin], 1);
```

This ensures that even if multiple threads try to update the same bin, each increment is correctly applied.

> **Platform note**: This lab targets **RDNA 3.5 wave32** (Radeon 8060S, gfx1151). Atomic contention is measured per 32-thread wavefront.

### Cost of Atomics

| Aspect | Impact |
|:-------|:-------|
| **Serialization** | Conflicting atomics are serialized |
| **Latency** | Higher latency than regular memory operations |
| **Contention** | Performance degrades with more conflicts |

## 3. Setup: Generate Test Data

In [ ]:
%%bash
# Generate test cases
python3 geninput.py
echo "Test cases generated in testcases/"
ls -la testcases/
echo ""
echo "Note: Large files (testcase 4: 100M elements) take time to read from disk."
echo "GPU advantage is most visible with large datasets and excludes file I/O time."

## 4. Compile All Implementations

In [ ]:
%%bash
# All three implementations now print "KERNEL_MS <ms>" on stderr
# (HIP events for GPU, std::chrono for serial). Recompile.
g++ -O2 fs_serial.cpp -o exe_serial
hipcc -O2 fs_bad.hip  -o exe_bad
hipcc -O2 fs_main.hip -o exe_optimized
echo "Compilation complete."

## 5. Approach 1: Naive Parallel Histogram (Bad Implementation)

Let us examine the naive approach in `fs_bad.hip`.

### Algorithm

Each block is responsible for counting **one specific bin**:

```cpp
__global__ void kernel(const int *input, int *histogram, int N, int num_bins) {
    __shared__ int sdata[1024];
    
    int tidx = threadIdx.x;
    int whichnumbins = blockIdx.x;  // Each block handles one bin
    
    sdata[tidx] = 0;
    
    // Each thread scans entire array looking for matches
    for (int i = tidx; i < N; i += blockDim.x) {
        if (input[i] == whichnumbins) 
            sdata[tidx]++;
    }
    __syncthreads();
    
    // Reduction to sum counts
    for (int i = blockDim.x / 2; i > 0; i >>= 1) {
        if (tidx < i) sdata[tidx] += sdata[tidx + i];
        __syncthreads();
    }
    
    if (tidx == 0) histogram[whichnumbins] = sdata[0];
}
```

### Exercise 1: Analyze the Problem

**Question 1**: How many times is each input element read from global memory?

Your answer: Each input element is read `num_bins` times from global memory, because every bin has one block that scans the whole input array.

**Question 2**: If num_bins = 256 and N = 10M, what is the total global memory traffic?

Your calculation: `N * num_bins = 10,000,000 * 256 = 2,560,000,000` global integer reads, which is about `2,560,000,000 * 4 B = 10.24 GB` of input-read traffic.

**Question 3**: Why is this approach inefficient?

Your answer: It repeats the same full-array scan once per bin, so the work and global memory reads are `O(N * num_bins)` instead of `O(N)`. Most reads only check values that do not match the block's bin, so memory bandwidth is wasted and the kernel scales poorly as the number of bins grows.

In [ ]:
%%bash
# Quick sanity run of the bad implementation
./exe_bad testcases/2.in > /dev/null && echo "Naive run OK"

## 6. Approach 2: Shared Memory with Atomics (Optimized)

The optimized implementation in `fs_main.hip` uses a two-level approach:

### Algorithm

```cpp
__global__ void kernel(const int* input, int* histogram, int N, int num_bins) {
    __shared__ int sdata[1024];  // Local histogram per block
    
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    int totalthread = blockDim.x * gridDim.x;

    // Step 1: Initialize shared memory histogram
    for (int i = threadIdx.x; i < num_bins; i += blockDim.x) {
        sdata[i] = 0;
    }
    __syncthreads();

    // Step 2: Accumulate into shared memory (local atomics)
    for (int i = idx; i < N; i += totalthread) {
        atomicAdd(&sdata[input[i]], 1);
    }
    __syncthreads();
    
    // Step 3: Merge local histograms into global (global atomics)
    for (int i = threadIdx.x; i < num_bins; i += blockDim.x) {
        atomicAdd(&histogram[i], sdata[i]);
    }
}
```

### Key Optimization: Two-Level Accumulation

| Level | Memory | Latency | Contention |
|:------|:-------|:--------|:-----------|
| Block-local | Shared Memory (LDS) | Low (~20 cycles) | Only within block |
| Global merge | Global Memory (HBM) | High (~400 cycles) | Reduced (one per block) |

### Exercise 2: Understand the Optimization

**Question 1**: Why is atomicAdd on shared memory faster than on global memory?

Your answer: Shared memory is on-chip and has much lower latency and higher bandwidth than global memory. Atomic conflicts are also limited to threads inside one block, while global-memory atomics must serialize updates through the much slower device memory path.

**Question 2**: In the merge step, how many global atomicAdd operations occur per bin?

Your answer: One global `atomicAdd` per block per bin. Across the whole grid, that is `gridDim.x` global atomic operations for each bin, instead of one global atomic for every input element.

**Question 3**: What is the purpose of the grid-stride loop pattern `for (int i = idx; i < N; i += totalthread)`?

Your answer: The grid-stride loop lets all launched threads cooperatively cover arrays larger than the total number of threads. Each thread processes indices separated by `blockDim.x * gridDim.x`, giving full coverage, balanced work, and coalesced access for neighboring threads.

In [ ]:
%%bash
# Quick sanity run of the optimized implementation + correctness check
./exe_optimized testcases/2.in > result_opt.txt
./exe_serial    testcases/2.in > result_serial.txt
diff -q result_serial.txt result_opt.txt && echo "Optimized matches serial"

## 7. Performance Comparison (real compute time)

The previous version measured **wall time** via `subprocess`, which was dominated by file I/O, GPU init, and stdout printing — the real differences between implementations were invisible.

Now each executable prints `KERNEL_MS <ms>` to stderr:
- `exe_serial`: `std::chrono` around `serial_histogram()` only (no file I/O)
- `exe_bad`, `exe_optimized`: HIP events around the GPU kernel (no H2D/D2H, no init)

We parse this number to get a fair compute-only comparison.

In [ ]:
import subprocess, re, statistics

# Multi-trial benchmark — parse REAL compute time from stderr (KERNEL_MS)
TRIALS = 5
KERN_RE = re.compile(r"KERNEL_MS\s+([0-9.eE+-]+)")

impls = [
    ("Serial CPU",    "./exe_serial"),
    ("Naive GPU",     "./exe_bad"),
    ("Optimized GPU", "./exe_optimized"),
]
testcases = [
    ("Test 2 (medium)", "testcases/2.in"),
    ("Test 4 (large)",  "testcases/4.in"),
]

def run_once(exe, path):
    r = subprocess.run([exe, path], capture_output=True, text=True)
    m = KERN_RE.search(r.stderr)
    if not m:
        raise RuntimeError(f"No KERNEL_MS from {exe} {path}: {r.stderr[:200]}")
    return float(m.group(1))

results = {}  # results[testcase][impl] = list of ms
for tc_name, tc_path in testcases:
    results[tc_name] = {}
    for impl_name, exe in impls:
        times = [run_once(exe, tc_path) for _ in range(TRIALS)]
        results[tc_name][impl_name] = times
        m, s = statistics.mean(times), statistics.stdev(times) if len(times) > 1 else 0.0
        print(f"{tc_name:<18} {impl_name:<15} mean={m:10.4f} ms  std={s:8.4f} ms")
    print()

### Exercise 3: Record Results

Fill in the execution times:

| Implementation | Time (testcase 2) | Time (testcase 4) |
|:---------------|:------------------|:------------------|
| Serial (CPU) | 2.3624 ms | 22.6050 ms |
| Naive GPU | 23.2188 ms | 2526.8180 ms |
| Optimized GPU | 0.4661 ms | 4.4517 ms |

Calculate speedups:

- Optimized vs Serial: 5.1x on testcase 2, 5.1x on testcase 4
- Optimized vs Naive: 49.8x on testcase 2, 567.6x on testcase 4

These timings are compute-only measurements from `KERNEL_MS`, excluding file I/O and output printing. The optimized GPU implementation is consistently faster than the serial baseline, and the advantage over the naive GPU version becomes much larger on the large testcase because the naive version rereads the full input once per bin.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

impl_names = [n for n, _ in impls]
tc_names   = list(results.keys())
colors     = ['#888888', '#dd8452', '#4c72b0']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: grouped bar chart per testcase (log scale)
x = np.arange(len(tc_names))
w = 0.27
for i, impl in enumerate(impl_names):
    means = [statistics.mean(results[tc][impl]) for tc in tc_names]
    stds  = [statistics.stdev(results[tc][impl]) if len(results[tc][impl]) > 1 else 0.0
             for tc in tc_names]
    bars = ax1.bar(x + (i - 1) * w, means, w, yerr=stds, capsize=4,
                   label=impl, color=colors[i], edgecolor='black')
    for b, m in zip(bars, means):
        ax1.text(b.get_x() + b.get_width()/2, m, f"{m:.2f}",
                 ha='center', va='bottom', fontsize=8)
ax1.set_xticks(x); ax1.set_xticklabels(tc_names)
ax1.set_ylabel("Compute Time (ms, log)")
ax1.set_yscale('log')
ax1.set_title(f"Histogram: Real Compute Time ({TRIALS} trials)")
ax1.legend(); ax1.grid(axis='y', alpha=0.3, which='both')

# Right: speedup of optimized over naive and serial
speedup_vs_serial = [statistics.mean(results[tc]["Serial CPU"]) /
                     statistics.mean(results[tc]["Optimized GPU"]) for tc in tc_names]
speedup_vs_naive  = [statistics.mean(results[tc]["Naive GPU"]) /
                     statistics.mean(results[tc]["Optimized GPU"]) for tc in tc_names]

w2 = 0.35
ax2.bar(x - w2/2, speedup_vs_serial, w2, label="Optimized / Serial",
        color='#55a467', edgecolor='black')
ax2.bar(x + w2/2, speedup_vs_naive,  w2, label="Optimized / Naive",
        color='#c44e52', edgecolor='black')
for i, (s1, s2) in enumerate(zip(speedup_vs_serial, speedup_vs_naive)):
    ax2.text(i - w2/2, s1, f"{s1:.1f}x", ha='center', va='bottom', fontsize=9)
    ax2.text(i + w2/2, s2, f"{s2:.1f}x", ha='center', va='bottom', fontsize=9)
ax2.set_xticks(x); ax2.set_xticklabels(tc_names)
ax2.set_ylabel("Speedup (x)")
ax2.set_yscale('log')
ax2.set_title("Speedup of Optimized GPU\n(true compute, excluding I/O)")
ax2.axhline(1, color='black', linewidth=0.8)
ax2.legend(); ax2.grid(axis='y', alpha=0.3, which='both')

plt.tight_layout()
plt.savefig("histogram_perf.png", dpi=150)
plt.show()
print("Saved: histogram_perf.png")
print()
print("=== Interpretation ===")
print("- Optimized GPU: each element read once + atomics on LDS -> very fast")
print("- Naive GPU:     each block scans entire input (N * num_bins reads) -> slow")
print("- Serial CPU:    O(N) but single-threaded -> baseline")
print("- Real speedups now visible (previously hidden by file I/O & stdout)")

## 8. Memory Access Pattern Analysis

### Naive Approach Memory Traffic

- Each block reads the **entire input array**
- Total reads = N x num_bins
- Example: N=10M, bins=256 → 2.56 billion reads

### Optimized Approach Memory Traffic

- Each element is read exactly **once** from global memory
- Total reads = N
- Example: N=10M → 10 million reads

### Exercise 4: Memory Efficiency

**Question**: Calculate the memory traffic reduction ratio:

$$
\text{Reduction} = \frac{\text{Naive reads}}{\text{Optimized reads}} = \frac{N \times \text{num\_bins}}{N} = ?
$$

Your answer: `num_bins`. Since the naive implementation reads `N * num_bins` input values and the optimized implementation reads only `N`, the reduction ratio is `(N * num_bins) / N = num_bins`. For example, with 256 bins the optimized approach reduces input global-memory reads by `256x`; with 512 bins it reduces them by `512x`.

## 9. Contention Analysis

### When Does Contention Occur?

Atomic contention happens when multiple threads try to update the same memory location simultaneously.

### Factors Affecting Contention

| Factor | Low Contention | High Contention |
|:-------|:---------------|:----------------|
| Number of bins | Many bins | Few bins |
| Data distribution | Uniform | Skewed |
| Memory level | Shared | Global |

### Exercise 5: Contention Scenarios

**Scenario 1**: All input values are the same (e.g., all zeros)

- What happens to performance? Performance becomes the worst among these distributions; the optimized kernel slows down because many threads update the same histogram bin.
- Why? All threads repeatedly execute `atomicAdd(&sdata[0], 1)`, so the shared-memory atomic operations serialize on one address. The two-level design still avoids most global atomic contention, but the block-local shared-memory contention is maximal.

**Scenario 2**: Input values are uniformly distributed across all bins

- Expected contention level: Low to moderate.
- Why? Updates are spread across all bins, so fewer threads target the same shared-memory address at the same time. The atomic operations are distributed more evenly, which reduces serialization and improves throughput.

In [ ]:
import numpy as np, os, subprocess, re, statistics
import matplotlib.pyplot as plt

# Atomic contention experiment — use REAL kernel time so the differences
# between distributions are not drowned in I/O.
os.makedirs("testcases_dist", exist_ok=True)
N, NUM_BINS = 10_000_000, 64
KERN_RE = re.compile(r"KERNEL_MS\s+([0-9.eE+-]+)")

def write_case(path, data):
    with open(path, "w") as f:
        f.write(f"{N} {NUM_BINS}\n")
        f.write(" ".join(map(str, data.tolist())))

rng = np.random.default_rng(0)
dists = {
    "Uniform":     rng.integers(0, NUM_BINS, N),
    "Gaussian":    np.clip(rng.normal(NUM_BINS/2, NUM_BINS/8, N).astype(int), 0, NUM_BINS-1),
    "All-same(0)": np.zeros(N, dtype=int),
}
paths = {n: f"testcases_dist/{n}.in" for n in dists}
for n, d in dists.items():
    if not os.path.exists(paths[n]):
        write_case(paths[n], d)

TRIALS = 5
dist_times = {}
for name, p in paths.items():
    ts = []
    for _ in range(TRIALS):
        r = subprocess.run(["./exe_optimized", p], capture_output=True, text=True)
        m = KERN_RE.search(r.stderr)
        if not m:
            raise RuntimeError(f"No KERNEL_MS for {p}")
        ts.append(float(m.group(1)))
    dist_times[name] = ts
    print(f"{name:<14} mean={statistics.mean(ts):8.4f} ms  std={statistics.stdev(ts):6.4f}")

means = [statistics.mean(dist_times[n]) for n in dists]
stds  = [statistics.stdev(dist_times[n]) for n in dists]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(list(dists.keys()), means, yerr=stds, capsize=5,
              color=['#55a467', '#4c72b0', '#c44e52'], edgecolor='black')
for b, m in zip(bars, means):
    ax.text(b.get_x() + b.get_width()/2, m, f"{m:.3f}",
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel("Kernel Time (ms)")
ax.set_title(f"Atomic Contention vs Data Distribution (real kernel time)\n"
             f"N={N:,}, bins={NUM_BINS}, optimized kernel")
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("histogram_contention.png", dpi=150)
plt.show()

print()
print("=== Interpretation ===")
print("- Uniform:     atomics spread across 64 bins -> minimal LDS contention")
print("- Gaussian:    most hits cluster near center bin -> moderate contention")
print("- All-same(0): every wave32 lane targets bin 0 -> maximum contention, slowest")
print("- The shared-memory privatization caps the cost: only LDS atomics collide,")
print("  not global atomics, so the slowdown is bounded.")

## 10. Summary

### Key Concepts

| Concept | Description |
|:--------|:------------|
| **Race Condition** | Multiple threads modifying same memory without synchronization |
| **Atomic Operation** | Indivisible read-modify-write operation |
| **Memory Contention** | Performance loss when atomics conflict frequently |
| **Privatization** | Using local (shared) memory to reduce global contention |

### Optimization Strategy

1. **Privatize** the histogram in shared memory (one per block)
2. Use **local atomics** (faster, less contention)
3. **Merge** block-local results into global histogram
4. Minimize global atomic operations

### When to Use Each Approach

| Scenario | Recommended Approach |
|:---------|:---------------------|
| Few bins, large N | Shared memory privatization |
| Many bins (> 1024) | May need multi-pass or sorting-based |
| Highly skewed data | Consider warp-level privatization |

## 11. Challenge Exercises

### Challenge 1: Implement Warp-Level Histogram

Modify the kernel to use warp-level privatization:
- Each warp maintains its own local histogram
- Reduces shared memory contention further

### Challenge 2: Benchmark with Different Distributions

Modify `geninput.py` to generate:
1. Uniform distribution
2. Gaussian distribution (most values near center)
3. Highly skewed (90% of values in one bin)

Compare performance for each distribution.

### Challenge 3: Vary Number of Bins

Test performance with num_bins = {16, 64, 256, 1024}

- How does bin count affect shared memory usage?
- At what point does the optimization become less effective?

---

## Lab Files Reference

| File | Description |
|:-----|:------------|
| `fs_serial.cpp` | CPU serial implementation (baseline) |
| `fs_bad.hip` | Naive GPU implementation (inefficient) |
| `fs_main.hip` | Optimized GPU implementation |
| `geninput.py` | Test case generator |
| `Makefile` | Build configuration |